# Терминал

Коваль:
https://www.youtube.com/watch?v=ys6VjUnWQCk
https://www.youtube.com/watch?v=U6CkCnJYhcU
https://www.youtube.com/watch?v=idUkymOQYJs
https://www.youtube.com/watch?v=OFhwMqXeQUE

Ахмедова:
https://www.youtube.com/watch?v=WC8MGi_sItE
https://www.youtube.com/watch?v=v1xb76zn2WQ

АРИАНА ЛОЛАЕВА:
https://www.youtube.com/watch?v=fdx7m7tgXdQ

Аранова Елизавета Варвара
https://www.youtube.com/watch?v=wAy_XfHCfuA

Слава Комиссаренко
https://www.youtube.com/watch?v=ExrG6AhR-wI
https://www.youtube.com/watch?v=k8Mixx5Q4vc

Алексей Квашонкин
https://www.youtube.com/watch?v=7GAr9pJeIp0

In [2]:
from pathlib import Path

url = "https://www.youtube.com/watch?v=OFhwMqXeQUE"


audio_path = "../data/!Notebook_test/audio/Idrak_first2.wav"
audio_name = Path(audio_path).stem

transcribe_path = "../data/!Notebook_test/transcripts_with_timestamp/"

print(transcribe_path + audio_name + ".json")

../data/!Notebook_test/transcripts_with_timestamp/Idrak_first2.json


# Тестирование отдельных функций

## Транскрибация локального аудиофайла в переменную (словарь)

In [3]:
import mlx_whisper

result = mlx_whisper.transcribe(
    str(audio_path),
    path_or_hf_repo="mlx-community/whisper-large-v3-turbo",
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
result

{'text': ' Какие у вас политические взгляды? Вот всегда-всегда вот такой нервный смешок. Но я вас понимаю, я тоже не распространяюсь о своих политических взглядах, но из-за того, что я немного медийный, обо мне иногда пишут, я про себя прочитал вот такое вот. Левацкое отродье. Вот так написали. И мне не понравилось, потому что я сам не уверен, кто я, решил загуглить, как понять, какие у тебя политические взгляды. И мне попалась там диаграмма политических взглядов. Вам попадалось такое? Диаграмма, по которой ты определяешь, кто ты. И оказалось, что я ультраправый радикал вообще. По многим взглядам совпало, например, отношение к гомосексуализму. У меня такое уже крайне негативное. Взять, если там лесбиянок, например. Я вообще не пойму, как можно хотеть женщин. Для меня это загадка. Или, например, отношение к эмигрантам. Я вот сам, как только, я три года назад переехал в Европу. Я как только переехал, я подумал, закрывайте ворота, нам эти обезьяны тут не нужны. Мы не для того, да, приехал

## Фильтрация словаря,получение только нужных колонок

In [10]:
filtered_result = {
    "text": result["text"],
    "segments": [
        {
            "id": segment["id"],
            "start": segment["start"],
            "end": segment["end"],
            "text": segment["text"],
        }
        for segment in result.get("segments", [])
    ],
}

In [11]:
filtered_result

{'text': ' Какие у вас политические взгляды? Вот всегда-всегда вот такой нервный смешок. Но я вас понимаю, я тоже не распространяюсь о своих политических взглядах, но из-за того, что я немного медийный, обо мне иногда пишут, я про себя прочитал вот такое вот. Левацкое отродье. Вот так вот. И мне не понравилось, потому что я сам не уверен, кто я, решил загуглить, как понять, какие у тебя политические взгляды. И мне попалась там диаграмма политических взглядов. Вам попадалось такое? Диаграмма, по которой ты определяешь, кто ты. И оказалось, что я ультраправый радикал вообще. По многим взглядам совпало, например, отношение к гомосексуализму. У меня такое уже крайне негативное. Взять, если там лесбиянок, например. Я вообще не пойму, как можно хотеть женщин. Для меня это загадка. Или, например, отношение к эмигрантам. Я вот сам, как только, я три года назад переехал в Европу. Я как только переехал, я подумал, закрывайте ворота, нам эти обезьяны тут не нужны. Мы не для того, да, приехали в Е

## Запись после фильтрация в json файл

In [17]:
import json

with open(transcribe_path + audio_name + ".json", "w", encoding="utf-8") as f:
    json.dump(filtered_result, f, ensure_ascii=False, indent=4)

## Чтение json файла в pandas DF

In [36]:
import json
import pandas as pd

# Загрузка JSON из файла или строки
with open(
    "../data/transcripts_with_timestamp/Node quietly dropped its biggest update in years....json",
    "r",
    encoding="utf-8",
) as f:
    data = json.load(f)

# Преобразуем ключ 'segments' в DataFrame
df = pd.DataFrame(data["segments"])

# Проверка результата
df.head()


,id,start,end,text
0,0,0.28,3.88,Node.js just did something huge and almost no...
1,1,3.88,8.46,"For the past decade, TypeScript has been repl..."
2,2,8.46,10.28,to back-end APIs.
3,3,10.28,13.24,"Everyone writes TypeScript now, whether they ..."
4,4,13.24,16.40,"And naturally, modern JavaScript runtimes had..."


## Работа с LLM

In [9]:
import os
from google import genai
import json


# Загрузка JSON из файла или строки
with open(
    "../data/transcripts_with_timestamp/Идрак Мирзализаде. Стендап концерт Это не я придумал.json",
    "r",
    encoding="utf-8",
) as f:
    data = json.load(f)

data = data["segments"]

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

prompt = f"""
Вы — эксперт по тематическому анализу аудио/видео транскриптов.

На входе вы получаете список объектов вида:

[
{{'id': 0,
  'start': 0.28,
  'end': 3.88,
  'text': 'Какие у вас политические взгляды?'}},
 {{'id': 1,
  'start': 3.88,
  'end': 8.46,
  'text': 'Вот всегда, всегда вот такой нервный смешок.'}}
]

Ваша задача:
1. Проанализировать все фрагменты текста и выделить глобальные ключевые темы (например, "политические взгляды", "медиа", "самоидентификация").
2. Для каждой темы определить временной диапазон: от минимальной до максимальной отметки `timestamp`, относящейся к сообщениям этой темы.
3. Вернуть результат **строго в формате JSON**, без каких-либо комментариев, префиксов, пояснений или форматирования.

Формат ожидаемого результата:
{{
"политические взгляды": {{"start_sec": 1.564, "end_sec": 3.530}},
"медиа": {{"start_sec": 8.951, "end_sec": 9.312}},
"самоидентификация": {{"start_sec": 16.077, "end_sec": 17.647}}
}}

Вот расшифровка:
---
{data}
---
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config={
        "response_mime_type": "application/json",
    },
)

response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        role='model'
      ),
      finish_reason=<FinishReason.MAX_TOKENS: 'MAX_TOKENS'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    prompt_token_count=45683,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=45683
      ),
    ],
    thoughts_token_count=65535,
    total_token_count=111218
  )
)

In [ ]:
formatted_text = json.loads(response.text)

formatted_text

{'Политические взгляды и Самоидентификация': {'start_sec': 0.0,
  'end_sec': 112.32},
 'Национализм и Стереотипы': {'start_sec': 112.32, 'end_sec': 264.96},
 'Расизм и Толерантность': {'start_sec': 264.96, 'end_sec': 488.68},
 'Свобода слова и Культура отмены': {'start_sec': 488.68, 'end_sec': 599.48},
 'Проблема тупости в обществе': {'start_sec': 598.48, 'end_sec': 967.48},
 'Отношения и Общественные Проблемы': {'start_sec': 967.48, 'end_sec': 1906.2},
 'Мужская анатомия и Мужские Проблемы': {'start_sec': 1908.32,
  'end_sec': 2939.32},
 'Вредные Привычки и Осознанность': {'start_sec': 2942.02, 'end_sec': 3032.8},
 'География, Жизнь за границей и Национальность': {'start_sec': 3033.26,
  'end_sec': 3105.8599999999997},
 'Такси, Стереотипы и Религия': {'start_sec': 3107.74, 'end_sec': 3152.72},
 'Комедия, Искусство и Взаимодействие с аудиторией': {'start_sec': 3160.02,
  'end_sec': 3268.12}}

# Скрапинг youtube для получения ссылок

In [ ]:
from yt_dlp import YoutubeDL

query = "стендап концерт"
search_url = f"https://www.youtube.com/results?search_query={query.replace(' ', '+')}"

ydl_opts = {
    "extract_flat": "in_playlist",
    "skip_download": True,
    "quiet": False,
    "simulate": True,
    "forceurl": True,
    "max_downloads": 10,  # ограничить число результатов
}
with YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(search_url, download=False)
    for entry in info.get("entries", []):
        print("https://www.youtube.com/watch?v=" + entry["id"])


https://www.youtube.com/watch?v=LV0I3_r9u-0
https://www.youtube.com/watch?v=c83NcqDvs-Y
https://www.youtube.com/watch?v=ExrG6AhR-wI
https://www.youtube.com/watch?v=M5z_OEoAOqE
https://www.youtube.com/watch?v=v1xb76zn2WQ
https://www.youtube.com/watch?v=1gAiMQkPHJc
https://www.youtube.com/watch?v=1nEDD1mQcmA
https://www.youtube.com/watch?v=di6tU8hQqSU
https://www.youtube.com/watch?v=fA-bcE6Z7eM
https://www.youtube.com/watch?v=bOzV0Okh2kw
https://www.youtube.com/watch?v=nf-s-6kppks
https://www.youtube.com/watch?v=w25BBF43seo
https://www.youtube.com/watch?v=qhx8QCDZ-cU
https://www.youtube.com/watch?v=c0ALcfrNsIQ
https://www.youtube.com/watch?v=mLGi0Si_OXU
https://www.youtube.com/watch?v=07diUFt0Weo
https://www.youtube.com/watch?v=hKu3APVhctU
https://www.youtube.com/watch?v=Xd8zyaVfp0Y
https://www.youtube.com/watch?v=td21ehWFfaM
https://www.youtube.com/watch?v=AzKFiKkJmAI
https://www.youtube.com/watch?v=X8k2WuraAoU
https://www.youtube.com/watch?v=CNGXS7O7kFg
https://www.youtube.com/watch?v=

In [8]:
import yt_dlp

URLS = "https://www.youtube.com/watch?v=LV0I3_r9u-0&t=503s"


class MyLogger:
    def debug(self, msg):
        # For compatibility with youtube-dl, both debug and info are passed into debug
        # You can distinguish them by the prefix '[debug] '
        if msg.startswith("[debug] "):
            pass
        else:
            self.info(msg)

    def info(self, msg):
        pass

    def warning(self, msg):
        pass

    def error(self, msg):
        print(msg)


# ℹ️ See "progress_hooks" in help(yt_dlp.YoutubeDL)
def my_hook(d):
    if d["status"] == "finished":
        print("Done downloading, now post-processing ...")


ydl_opts = {
    "logger": MyLogger(),
    "progress_hooks": [my_hook],
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download(URLS)


Done downloading, now post-processing ...
Done downloading, now post-processing ...


# Скачивание датасетов с HF

In [11]:
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("RayYuki/CodecBench_laughterscape_ver1.0", split="train")

# В DataFrame
df = pd.DataFrame(dataset)
df

,filename,audio,sample_rate,duration
0,spkr063-utt013.wav,"{'path': 'spkr063-utt013.wav', 'array': [0.001...",24000,0.510042
1,spkr377-utt003.wav,"{'path': 'spkr377-utt003.wav', 'array': [-0.00...",24000,1.220042
2,spkr223-utt006.wav,"{'path': 'spkr223-utt006.wav', 'array': [-0.00...",24000,2.110042
3,spkr371-utt005.wav,"{'path': 'spkr371-utt005.wav', 'array': [-0.00...",24000,0.900042
4,spkr272-utt050.wav,"{'path': 'spkr272-utt050.wav', 'array': [0.000...",24000,0.922042
...,...,...,...,...
8165,spkr148-utt004.wav,"{'path': 'spkr148-utt004.wav', 'array': [0.001...",24000,1.420000
8166,spkr356-utt007.wav,"{'path': 'spkr356-utt007.wav', 'array': [-0.00...",24000,0.586042
8167,spkr085-utt180.wav,"{'path': 'spkr085-utt180.wav', 'array': [-0.01...",24000,0.639042
8168,spkr212-utt020.wav,"{'path': 'spkr212-utt020.wav', 'array': [-3.05...",24000,2.615042


# SWIFT test

Требуется редактирование swift кода и последующая компеляция командой:
swiftc main.swift -o SoundFileClassifier

## Усович

In [ ]:
# # Тест нормализации звука

# from pydub import AudioSegment, effects

# # Загружаем аудио
# audio = AudioSegment.from_wav(
#     "/Users/aleksandr/Documents/Projects/StandUP_project/data/audio/Ваня Усович ＂ЕЩЕ ОДИН ДЕНЬ＂ 2020.wav"
# )

# # 1. Выравнивание уровня громкости (RMS normalization)
# normalized_audio = effects.normalize(audio)

# # 2. Применение компрессии для уменьшения разницы между громкой речью и тихим смехом
# compressed_audio = normalized_audio.compress_dynamic_range()

# # Сохранение
# compressed_audio.export(
#     "/Users/aleksandr/Documents/Projects/StandUP_project/data/audio/Ваня Усович ＂ЕЩЕ ОДИН ДЕНЬ＂ 2020_normalized_output.wav",
#     format="wav",
# )


<_io.BufferedRandom name='/Users/aleksandr/Documents/Projects/StandUP_project/data/audio/Ваня Усович ＂ЕЩЕ ОДИН ДЕНЬ＂ 2020_normalized_output.wav'>

In [ ]:
# Список аргументов для передачи swift приложению
# Usage: SoundFileClassifier <input_audio_path.wav> <output_json_path.json> <window_duration_seconds> <preferred_timescale> <confidence_threshold>

import subprocess

audio_path = "/Users/aleksandr/Documents/Projects/StandUP_project/data/audio/Ваня Усович ＂ЕЩЕ ОДИН ДЕНЬ＂ 2020_normalized_output.wav"
output_path = "/Users/aleksandr/Documents/Projects/StandUP_project/data/laughter_segmentation_json/Ivan_normalized_output.json"
window_duration_seconds = "0.5"  # min - 0.5
preferred_timescale = "1000"  # дробная часть секунды 1 / preferred_timescale
confidence_threshold = "0.25"  # определение нижней границы чувствительности смеха

subprocess.run(
    [
        "/Users/aleksandr/Documents/Projects/StandUP_project/src/SoundFileClassifier",  # путь до swift бинарника
        audio_path,
        output_path,
        window_duration_seconds,
        preferred_timescale,
        confidence_threshold,
    ]
)

The request completed successfully!
Результаты анализа сохранены в: /Users/aleksandr/Documents/Projects/StandUP_project/data/laughter_segmentation_json/Ivan_normalized_output.json


CompletedProcess(args=['/Users/aleksandr/Documents/Projects/StandUP_project/src/SoundFileClassifier', '/Users/aleksandr/Documents/Projects/StandUP_project/data/audio/Ваня Усович ＂ЕЩЕ ОДИН ДЕНЬ＂ 2020_normalized_output.wav', '/Users/aleksandr/Documents/Projects/StandUP_project/data/laughter_segmentation_json/Ivan_normalized_output.json', '0.5', '1000', '0.25'], returncode=0)

In [12]:
import json
import pandas as pd

# Загрузка JSON из файла
with open("../data/laughter_segmentation_json/Ivan_normalized_output.json", "r") as f:
    data = json.load(f)

# Преобразование в DataFrame
df_ivan = pd.DataFrame(list(data.items()), columns=["time", "confidence"])

# Преобразование time в float (если требуется числовая ось)
df_ivan["time"] = df_ivan["time"].astype(float)

# Сортировка по времени (опционально)
df_ivan.sort_values("time", inplace=True)

# Результат
df_ivan


,time,confidence
245,2.75,0.282760
638,4.25,0.268598
724,5.75,0.303010
1059,6.25,0.322871
1354,6.75,0.272182
...,...,...
31,3242.00,0.279420
1643,3243.00,0.259268
900,3243.25,0.311682
982,3244.50,0.253812


## Идрак

In [ ]:
# Usage: SoundFileClassifier <input_audio_path.wav> <output_json_path.json> <window_duration_seconds> <preferred_timescale> <confidence_threshold>

import subprocess

audio_path = "/Users/aleksandr/Documents/Projects/StandUP_project/data/audio/Идрак Мирзализаде. Стендап концерт Это не я придумал.wav"
output_path = "/Users/aleksandr/Documents/Projects/StandUP_project/data/laughter_segmentation_json/Idrak.json"
window_duration_seconds = "0.5"  # min - 0.5
preferred_timescale = "1000"  # дробная часть секунды 1 / preferred_timescale
confidence_threshold = "0.25"  # определение нижней границы чувствительности смеха

subprocess.run(
    [
        "/Users/aleksandr/Documents/Projects/StandUP_project/src/SoundFileClassifier",  # путь до бинарника
        audio_path,
        output_path,
        window_duration_seconds,
        preferred_timescale,
        confidence_threshold,
    ]
)


The request completed successfully!
Результаты анализа сохранены в: /Users/aleksandr/Documents/Projects/StandUP_project/data/laughter_segmentation_json/Idrak.json


CompletedProcess(args=['/Users/aleksandr/Documents/Projects/StandUP_project/src/SoundFileClassifier', '/Users/aleksandr/Documents/Projects/StandUP_project/data/audio/Идрак Мирзализаде. Стендап концерт Это не я придумал.wav', '/Users/aleksandr/Documents/Projects/StandUP_project/data/laughter_segmentation_json/Idrak.json', '1', '1000', '0.3'], returncode=0)

In [21]:
import json
import pandas as pd

# Загрузка JSON из файла
with open("../data/laughter_segmentation_json/Idrak.json", "r") as f:
    data = json.load(f)

# Преобразование в DataFrame
df_idrak = pd.DataFrame(list(data.items()), columns=["time", "confidence"])

# Преобразование time в float (если требуется числовая ось)
df_idrak["time"] = df_idrak["time"].astype(float)

# Сортировка по времени (опционально)
df_idrak.sort_values("time", inplace=True)

# Результат
df_idrak


,time,confidence
1488,1.5,0.547360
2023,2.5,0.622392
1267,3.0,0.805686
1539,8.5,0.512357
2011,16.0,0.761802
...,...,...
413,3260.5,0.779873
2399,3261.0,0.580877
24,3261.5,0.615865
2422,3262.0,0.447715


## Дима

In [ ]:
# Usage: SoundFileClassifier <input_audio_path.wav> <output_json_path.json> <window_duration_seconds> <preferred_timescale> <confidence_threshold>

import subprocess

audio_path = "/Users/aleksandr/Documents/Projects/StandUP_project/data/audio/Дима Коваль. Стендап концерт из Вегаса..wav"
output_path = "/Users/aleksandr/Documents/Projects/StandUP_project/data/laughter_segmentation_json/Dima.json"
window_duration_seconds = "0.5"  # min - 0.5
preferred_timescale = "1000"  # дробная часть секунды 1 / preferred_timescale
confidence_threshold = "0.25"  # определение нижней границы чувствительности смеха

subprocess.run(
    [
        "/Users/aleksandr/Documents/Projects/StandUP_project/src/SoundFileClassifier",  # путь до бинарника
        audio_path,
        output_path,
        window_duration_seconds,
        preferred_timescale,
        confidence_threshold,
    ]
)


The request completed successfully!
Результаты анализа сохранены в: /Users/aleksandr/Documents/Projects/StandUP_project/data/laughter_segmentation_json/Dima.json


CompletedProcess(args=['/Users/aleksandr/Documents/Projects/StandUP_project/src/SoundFileClassifier', '/Users/aleksandr/Documents/Projects/StandUP_project/data/audio/Дима Коваль. Стендап концерт из Вегаса..wav', '/Users/aleksandr/Documents/Projects/StandUP_project/data/laughter_segmentation_json/Dima.json', '1', '1000', '0.3'], returncode=0)

In [23]:
import json
import pandas as pd

# Загрузка JSON из файла
with open("../data/laughter_segmentation_json/Dima.json", "r") as f:
    data = json.load(f)

# Преобразование в DataFrame
df_idrak = pd.DataFrame(list(data.items()), columns=["time", "confidence"])

# Преобразование time в float (если требуется числовая ось)
df_idrak["time"] = df_idrak["time"].astype(float)

# Сортировка по времени (опционально)
df_idrak.sort_values("time", inplace=True)

# Результат
df_idrak


,time,confidence
386,28.5,0.418664
104,29.0,0.523689
110,29.5,0.316232
22,31.0,0.347578
146,43.0,0.552869
...,...,...
268,1058.0,0.366513
195,1064.0,0.909561
465,1064.5,0.824979
500,1065.0,0.707863
